[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/05_awq_benchmark.ipynb)

# 05 · AWQ 4-bit + GPU benchmark (16-bit vs 8-bit vs AWQ)

Quantises the merged model with AutoAWQ (4-bit GEMM, group 128, calibrated on *training* postings) and benchmarks 16-bit, bitsandbytes 8-bit and AWQ on the same evenly spaced test subset at batch size 1: latency, tokens/s, peak VRAM, size on disk and accuracy. AutoAWQ is archived upstream, so it gets its own pinned environment; if it fails to install on your Colab image, the AWQ row stays *pending* and the other rows still run. Same code as `make local-awq` (`jobx.pipeline.awq_benchmark`).

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~45 min**. Cells after the setup are safe to re-run.

**Outputs:** `results/benchmarks/colab_t4.json`, predictions + metrics for `qwen2.5-1.5b_merged_int8` and `qwen2.5-1.5b_awq` (pushed to `<HF_USERNAME>/jobx-artifacts`); AWQ model pushed to `<HF_USERNAME>/jobx-qwen2.5-1.5b-awq`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q autoawq==0.2.9 transformers==4.51.3 accelerate==1.10.1 huggingface_hub==0.35.3 datasets==4.1.1 bitsandbytes==0.48.1
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

In [ ]:
import json
from jobx.pipeline import awq_benchmark, load_profile
PROFILE = load_profile('colab_t4')   # n_bench = 150
bench_file = awq_benchmark(PROFILE, push=True)
print(json.dumps(json.loads(bench_file.read_text()), indent=2))